# 01 · Explore SQuAD 2.0 and try a ready-made model

**Task T2** in [TASKS.md](../TASKS.md). Run this top to bottom in Google Colab (Runtime → Change runtime type → **T4 GPU**).

What this notebook does:
1. Loads SQuAD 2.0 and counts answerable vs. unanswerable questions
2. Shows real examples, question types and answer lengths
3. Measures passage lengths in tokens, to see how often the 384-token window is too short
4. Runs the published `deepset/roberta-base-squad2` model on 50 dev questions as a baseline
5. Runs the preprocessing sanity check
6. Saves the key numbers to `results/data_stats.md` for the report

## 0. Setup
Clones the repo and installs the requirements in Colab. Safe to run more than once.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Davidfaleye360/AI-Question-Answering-System.git"
REPO_DIR = "/content/AI-Question-Answering-System"

if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    %cd {REPO_DIR}
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # running locally: work from the repo root

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import transformers, datasets, torch

print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)")
if int(transformers.__version__.split(".")[0]) >= 5:
    raise RuntimeError("transformers 5.x is still loaded. Use Runtime -> Restart session, then run the setup cell again.")

In [ ]:
from src.config import load_config
from src.data import load_squad

cfg = load_config("configs/distilbert.yaml")
raw = load_squad(cfg.data.dataset_name)
raw

## 1. Size and balance
The dev set (`validation`) is our test set, because the official SQuAD 2.0 test set is hidden.

In [ ]:
import pandas as pd
from src.explore import dataset_summary

summary = pd.DataFrame.from_dict({split: dataset_summary(raw[split]) for split in ["train", "validation"]}, orient="index")
summary

**What to notice:** about a third of training questions are unanswerable, but about half of dev questions are. That's why we always report answerable (HasAns) and unanswerable (NoAns) scores separately.

## 2. What the data looks like

In [ ]:
def show(ex):
    ans = ex["answers"]["text"]
    print("QUESTION:", ex["question"])
    print("ANSWER:  ", repr(ans[0]) if ans else "(no answer)",
          "| answer_start =", ex["answers"]["answer_start"][0] if ans else "-")
    print("PASSAGE: ", ex["context"][:600], "..." if len(ex["context"]) > 600 else "")
    print("-" * 100)

train = raw["train"]
show(train[0])
first_unanswerable = next(i for i in range(len(train)) if len(train[i]["answers"]["text"]) == 0)
show(train[first_unanswerable])

`answer_start` is a **character** position in the passage. Check it yourself:

In [ ]:
ex = train[0]
s = ex["answers"]["answer_start"][0]
ex["context"][s : s + len(ex["answers"]["text"][0])]

## 3. Question types

In [ ]:
import matplotlib.pyplot as plt
from src.explore import question_type_counts

qtypes = pd.Series(question_type_counts(raw["validation"])).sort_values()
ax = qtypes.plot.barh(color="#2a78d6", figsize=(7, 4), width=0.7)
ax.set_title("Dev set questions by type")
ax.set_xlabel("questions")
ax.grid(axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
(qtypes / qtypes.sum() * 100).round(1).sort_values(ascending=False)

We'll reuse these buckets in the error analysis (T8) to see which kinds of question the model finds hardest.

## 4. Lengths: do passages fit in 384 tokens?

The next cell prints a warning like "sequence length is longer than the specified maximum (718 > 512)". It's harmless here: we are only counting tokens, not running the model. During preprocessing, long passages like that one are split into overlapping windows.

In [ ]:
from transformers import AutoTokenizer
from src.explore import answer_lengths_words, percentile, sample_rows, share_needing_windows, token_lengths

tokenizer = AutoTokenizer.from_pretrained(cfg.model.checkpoint)
rows = sample_rows(raw["train"], 3000, seed=42)
q_lens, c_lens = token_lengths(rows, tokenizer)
share = share_needing_windows(q_lens, c_lens, cfg.preprocessing.max_length)

print(f"Passage tokens: median {percentile(c_lens, 50):.0f}, 90th pct {percentile(c_lens, 90):.0f}, max {max(c_lens)}")
print(f"Question tokens: median {percentile(q_lens, 50):.0f}, max {max(q_lens)}")
print(f"Examples needing more than one {cfg.preprocessing.max_length}-token window: {share:.1%}")

ax = pd.Series(c_lens).plot.hist(bins=50, color="#2a78d6", figsize=(7, 3.5))
ax.axvline(cfg.preprocessing.max_length, color="#eb6834", linewidth=2)
ax.text(cfg.preprocessing.max_length + 8, ax.get_ylim()[1] * 0.85, f"{cfg.preprocessing.max_length}-token limit")
ax.set_title("Passage length in tokens (3,000 training examples)")
ax.set_xlabel("tokens")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
ans_words = answer_lengths_words(raw["validation"])
print(f"Answer length in words: median {percentile(ans_words, 50):.0f}, 90th pct {percentile(ans_words, 90):.0f}, max {max(ans_words)}")
print(f"Answers of 10 words or fewer: {sum(w <= 10 for w in ans_words) / len(ans_words):.1%}")

Answers are short, which is why post-processing only considers spans up to `max_answer_length = 30` tokens.

## 5. Baseline: a model someone else already trained

`deepset/roberta-base-squad2` was fine-tuned on SQuAD 2.0 by deepset. Its published dev scores are **79.9 EM / 82.9 F1**.
Here we run it on 50 random dev questions, just to see the task working end to end. 50 questions is a small sample, so expect the scores to differ from the published ones.

In [ ]:
import torch
from transformers import pipeline

qa = pipeline("question-answering", model="deepset/roberta-base-squad2",
              device=0 if torch.cuda.is_available() else -1)

dev_sample = sample_rows(raw["validation"], 50, seed=7)
preds = qa(question=[r["question"] for r in dev_sample], context=[r["context"] for r in dev_sample],
           handle_impossible_answer=True, max_answer_len=30)

table = pd.DataFrame({
    "question": [r["question"] for r in dev_sample],
    "gold": [r["answers"]["text"][0] if r["answers"]["text"] else "(no answer)" for r in dev_sample],
    "predicted": [p["answer"] or "(no answer)" for p in preds],
    "score": [round(p["score"], 3) for p in preds],
})
table.head(15)

In [ ]:
import evaluate

squad_v2 = evaluate.load("squad_v2")
baseline = squad_v2.compute(
    predictions=[{"id": r["id"], "prediction_text": p["answer"], "no_answer_probability": 0.0}
                 for r, p in zip(dev_sample, preds)],
    references=[{"id": r["id"], "answers": r["answers"]} for r in dev_sample],
)
{k: round(v, 1) for k, v in baseline.items() if k in ("exact", "f1", "HasAns_f1", "NoAns_f1", "HasAns_total", "NoAns_total")}

**Try it yourself:** change the question and passage below. Ask something the passage can't answer and see if the model says nothing.

In [ ]:
qa(question="When did Curie win the Chemistry prize?",
   context="Marie Curie won the Nobel Prize in Physics in 1903 and in Chemistry in 1911.",
   handle_impossible_answer=True)

## 6. Preprocessing sanity check
Turns every training label back into text and compares it with the gold answer. It must print **RESULT: PASS** before we train (T3).

In [ ]:
# {sys.executable} is the Python this notebook runs on, so the script sees the same installed packages.
!{sys.executable} scripts/check_preprocessing.py --n-examples 2000 --n-show 15

## 7. Save the numbers for the report

In [ ]:
from pathlib import Path

lines = ["# SQuAD 2.0 data statistics", "", "Generated by `notebooks/01_explore_data.ipynb`.", "",
         "| Split | Questions | Answerable | Unanswerable | Unanswerable % | Passages | Articles |",
         "| --- | --- | --- | --- | --- | --- | --- |"]
for r in summary.itertuples():
    lines.append(f"| {r.Index} | {r.questions:,} | {r.answerable:,} | {r.unanswerable:,} | {r.unanswerable_pct} | {r.passages:,} | {r.articles:,} |")
lines += ["",
          f"- Passage tokens (3,000 train samples): median {percentile(c_lens, 50):.0f}, 90th percentile {percentile(c_lens, 90):.0f}, max {max(c_lens)}",
          f"- Examples needing more than one {cfg.preprocessing.max_length}-token window: {share:.1%}",
          f"- Answer length in words (dev): median {percentile(ans_words, 50):.0f}, 90th percentile {percentile(ans_words, 90):.0f}",
          f"- RoBERTa baseline on 50 random dev questions: EM {baseline['exact']:.1f}, F1 {baseline['f1']:.1f}",
          "", "## Dev questions by type (%)", ""]
lines += [f"- {k}: {v}" for k, v in (qtypes / qtypes.sum() * 100).round(1).sort_values(ascending=False).items()]

Path("results").mkdir(exist_ok=True)
Path("results/data_stats.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## Next
- Commit `results/data_stats.md` (Colab: download it, or push from a local clone).
- Tick **T2** in TASKS.md.
- Move on to **T3: smoke-test training**.